<a href="https://colab.research.google.com/github/Najla-Nuricia/PemMes_244107020091_Najla/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 16.2 MB/s  0:00:01
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=97cd69f4c639fd0106b5300e4b0a12344dd3879ab7b6fac4fc1145c482714e2c
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=0154f215711823f9ee84fb1321cc2a1374f4ed078cc9dce3e2fc25e49bf1b853
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [3]:
import numpy as np
import pandas as pd
import time

from annoy import AnnoyIndex
import faiss
import hnswlib

# Konfigurasi eksperimen
DATA_CONFIGS = [
    (1000, 2),
    (1000, 5),
    (1000000, 2),
    (1000000, 5)
]

METRICS = ["euclidean", "cosine"]

K = 10
N_QUERY = 100

results = []


# Normalisasi data untuk cosine similarity
def normalize(data):
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-10)


# Menghitung ground truth menggunakan brute force
def get_ground_truth(data, queries, metric):
    if metric == "cosine":
        data = normalize(data)
        queries = normalize(queries)

    ground_truth = []

    for q in queries:
        if metric == "euclidean":
            distances = np.sum((data - q) ** 2, axis=1)
            nearest = np.argpartition(distances, K)[:K]
        else:
            similarities = data @ q
            nearest = np.argpartition(-similarities, K)[:K]

        ground_truth.append(nearest)

    return np.array(ground_truth)


# Menghitung recall
def calculate_recall(predictions, ground_truth):
    recalls = []

    for pred, truth in zip(predictions, ground_truth):
        recall = len(set(pred) & set(truth)) / K
        recalls.append(recall)

    return np.mean(recalls)


# ANNOY
def test_annoy(data, queries, metric):
    dimension = data.shape[1]

    annoy_metric = (
        "euclidean" if metric == "euclidean" else "angular"
    )

    index = AnnoyIndex(dimension, annoy_metric)

    start = time.perf_counter()

    for i, vector in enumerate(data):
        index.add_item(i, vector.tolist())

    index.build(10)

    build_time = time.perf_counter() - start

    predictions = []

    start = time.perf_counter()

    for q in queries:
        ids = index.get_nns_by_vector(
            q.tolist(), K, search_k=-1
        )
        predictions.append(ids)

    query_time = (
        time.perf_counter() - start
    ) / len(queries)

    return build_time, query_time, np.array(predictions)


# FAISS
def test_faiss(data, queries, metric):
    dimension = data.shape[1]

    if metric == "euclidean":
        index = faiss.IndexHNSWFlat(
            dimension, 16, faiss.METRIC_L2
        )
    else:
        index = faiss.IndexHNSWFlat(
            dimension, 16, faiss.METRIC_INNER_PRODUCT
        )

        data = normalize(data).astype("float32")
        queries = normalize(queries).astype("float32")

    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 64

    start = time.perf_counter()
    index.add(data)
    build_time = time.perf_counter() - start

    start = time.perf_counter()
    _, predictions = index.search(queries, K)
    query_time = (
        time.perf_counter() - start
    ) / len(queries)

    return build_time, query_time, predictions


# HNSW
def test_hnsw(data, queries, metric):
    dimension = data.shape[1]

    space = "l2" if metric == "euclidean" else "cosine"

    if metric == "cosine":
        data = normalize(data).astype("float32")
        queries = normalize(queries).astype("float32")

    index = hnswlib.Index(
        space=space,
        dim=dimension
    )

    start = time.perf_counter()

    index.init_index(
        max_elements=len(data),
        ef_construction=200,
        M=16
    )

    index.add_items(data)

    build_time = time.perf_counter() - start

    index.set_ef(64)

    start = time.perf_counter()
    predictions, _ = index.knn_query(queries, k=K)

    query_time = (
        time.perf_counter() - start
    ) / len(queries)

    return build_time, query_time, predictions


# Menjalankan seluruh eksperimen
for n_data, dimension in DATA_CONFIGS:

    print(f"\nDataset: {n_data:,} data | {dimension}D")

    rng = np.random.default_rng(42)

    data = rng.random(
        (n_data, dimension),
        dtype=np.float32
    )

    queries = rng.random(
        (N_QUERY, dimension),
        dtype=np.float32
    )

    ground_truth = get_ground_truth(
        data, queries, "euclidean"
    )

    algorithms = {
        "ANNOY": test_annoy,
        "FAISS": test_faiss,
        "HNSW": test_hnsw
    }

    for name, algorithm in algorithms.items():

        build, query, predictions = algorithm(
            data.copy(),
            queries.copy(),
            "euclidean"
        )

        recall = calculate_recall(
            predictions,
            ground_truth
        )

        results.append({
            "Jumlah Data": n_data,
            "Dimensi": f"{dimension}D",
            "Algoritma": name,
            "Build Time (s)": round(build, 4),
            "Query Time (ms)": round(query * 1000, 4),
            "Recall@10 (%)": round(recall * 100, 2)
        })

        print(
            f"{name}: Build={build:.4f}s | "
            f"Query={query*1000:.4f}ms | "
            f"Recall={recall*100:.2f}%"
        )

df = pd.DataFrame(results)
display(df)

df.to_csv("hasil_perbandingan_ann.csv", index=False)


# Menampilkan hasil
df = pd.DataFrame(results)

print("\n=== HASIL EKSPERIMEN ===")
display(df)

# Simpan hasil ke CSV
df.to_csv("hasil_perbandingan_ann.csv", index=False)

print("\nHasil disimpan ke hasil_perbandingan_ann.csv")


Dataset: 1,000 data | 2D
ANNOY: Build=0.0207s | Query=0.0305ms | Recall=99.90%
FAISS: Build=0.0719s | Query=0.0179ms | Recall=100.00%
HNSW: Build=0.0558s | Query=0.0158ms | Recall=100.00%

Dataset: 1,000 data | 5D
ANNOY: Build=0.0146s | Query=0.0304ms | Recall=96.80%
FAISS: Build=0.0995s | Query=0.0231ms | Recall=100.00%
HNSW: Build=0.0698s | Query=0.0273ms | Recall=100.00%

Dataset: 1,000,000 data | 2D
ANNOY: Build=27.4412s | Query=0.0851ms | Recall=99.00%
FAISS: Build=129.3333s | Query=0.0600ms | Recall=100.00%
HNSW: Build=124.4375s | Query=0.0466ms | Recall=100.00%

Dataset: 1,000,000 data | 5D
ANNOY: Build=21.5560s | Query=0.0894ms | Recall=95.30%
FAISS: Build=203.1277s | Query=0.0597ms | Recall=100.00%
HNSW: Build=191.3559s | Query=0.0594ms | Recall=100.00%


,Jumlah Data,Dimensi,Algoritma,Build Time (s),Query Time (ms),Recall@10 (%)
0,1000,2D,ANNOY,0.0207,0.0305,99.9
1,1000,2D,FAISS,0.0719,0.0179,100.0
2,1000,2D,HNSW,0.0558,0.0158,100.0
3,1000,5D,ANNOY,0.0146,0.0304,96.8
4,1000,5D,FAISS,0.0995,0.0231,100.0
5,1000,5D,HNSW,0.0698,0.0273,100.0
6,1000000,2D,ANNOY,27.4412,0.0851,99.0
7,1000000,2D,FAISS,129.3333,0.0600,100.0
8,1000000,2D,HNSW,124.4375,0.0466,100.0
9,1000000,5D,ANNOY,21.5560,0.0894,95.3



=== HASIL EKSPERIMEN ===


,Jumlah Data,Dimensi,Algoritma,Build Time (s),Query Time (ms),Recall@10 (%)
0,1000,2D,ANNOY,0.0207,0.0305,99.9
1,1000,2D,FAISS,0.0719,0.0179,100.0
2,1000,2D,HNSW,0.0558,0.0158,100.0
3,1000,5D,ANNOY,0.0146,0.0304,96.8
4,1000,5D,FAISS,0.0995,0.0231,100.0
5,1000,5D,HNSW,0.0698,0.0273,100.0
6,1000000,2D,ANNOY,27.4412,0.0851,99.0
7,1000000,2D,FAISS,129.3333,0.0600,100.0
8,1000000,2D,HNSW,124.4375,0.0466,100.0
9,1000000,5D,ANNOY,21.5560,0.0894,95.3



Hasil disimpan ke hasil_perbandingan_ann.csv
